# Notebook 08 — Extended Analysis: Ring Attractor, CTG, Cross-Subject Transfer
**Pipeline position:** After notebooks 01–07.

**Input:** `results/01_epochs_{subj}.npz`, `results/02_geometry_{subj}.npz`  
**Output:** `results/08_extended_results.json`, `results/08_ring_sigma.npz`

Four analyses:
1. **Ring attractor phase:** Does the maintenance manifold have circular topology (Compte et al. 2000)?
2. **Cross-temporal generalisation (CTG):** Is the maintenance code time-invariant (King & Dehaene 2014)?
3. **Cross-subject geometry transfer:** Does Procrustes-aligned geometry generalise across subjects?
4. **σ₁/σ₂ quantitative test:** Ratio of top-2 singular values as a ring-attractor metric across WM loads.

**References:** Compte et al. (2000) *Cereb Cortex*; King & Dehaene (2014) *Psychol Sci*; Wimmer et al. (2014) *Nat Neurosci*; Bernardi et al. (2020) *Cell*.


In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy import stats

sys.path.insert(0, str(Path("../src")))

from geometry import latent_trajectories, cross_temporal_generalization, subspace_overlap
from dynamics import trajectory_tangling, ring_attractor_phase, velocity_autocorrelation
from neuroai import centered_kernel_alignment, procrustes_align
from statistics import temporal_cluster_permutation
from preprocessing import SUBJECTS
from visualization import nature_style, PALETTE, save_figure

nature_style()
RESULTS   = Path("../results")
MAINT_WIN = (0.3, 1.4)   # maintenance window (s)
N_LATENT  = 6            # latent dims to use from saved Z

print(f"Extended analysis — subjects: {SUBJECTS}")


In [ ]:
# Load preprocessed data from previous notebooks
epochs_by_subj = {}   # {subj: (N, C, T)} — preprocessed HGP from nb01
Z_by_subj      = {}   # {subj: (N, T_lat, k)} — latent trajectories from nb02
labels_by_subj = {}   # {subj: (N,) task_id 0/1/2-back}
times_by_subj  = {}   # {subj: (T,) time axis}

for subj in SUBJECTS:
    ep_path  = RESULTS / f"01_epochs_{subj}.npz"
    geo_path = RESULTS / f"02_geometry_{subj}.npz"
    if not ep_path.exists() or not geo_path.exists():
        print(f"[{subj}] results missing — run notebooks 01 and 02 first")
        continue
    ep  = np.load(ep_path)
    geo = np.load(geo_path, allow_pickle=True)
    epochs_by_subj[subj] = ep["epochs"]    # (N, C, T)
    Z_by_subj[subj]      = geo["Z"]        # (N, T_lat, k)
    labels_by_subj[subj] = ep["task_id"]
    times_by_subj[subj]  = ep["times"]
    print(f"[{subj}] epochs {ep['epochs'].shape}  Z {geo['Z'].shape}")

SUBJECTS_OK = list(Z_by_subj.keys())
times_ref   = times_by_subj[SUBJECTS_OK[0]] if SUBJECTS_OK else None
maint_mask  = (times_ref >= MAINT_WIN[0]) & (times_ref <= MAINT_WIN[1]) if times_ref is not None else None
print(f"\nSubjects loaded: {SUBJECTS_OK}")


## Analysis 1: Ring Attractor Phase
**Hypothesis (Compte et al. 2000; Wimmer et al. 2014):** The top-2D PCA projection of the 2-back maintenance trajectory traces a circular manifold. Phase angle φ(t) = arctan2(PC₂, PC₁) should drift slowly (low tangling) and show circular topology.
**Falsification:** High tangling or random phase ↔ non-circular attractor.


In [ ]:
# Analysis 1: ring attractor phase from latent trajectories (nb02)
ring_results = {}

for subj in SUBJECTS_OK:
    Z      = Z_by_subj[subj]          # (N_trials, T, k)
    labels = labels_by_subj[subj]
    times  = times_by_subj[subj]

    mask_2back = labels == 2
    if mask_2back.sum() < 10:
        print(f"[{subj}] Insufficient 2-back trials ({mask_2back.sum()}), skipping")
        continue

    maint_t = (times >= MAINT_WIN[0]) & (times <= MAINT_WIN[1])
    # Trial-averaged latent trajectory during maintenance
    Z_pca = Z[mask_2back][:, maint_t, :N_LATENT].mean(0)   # (T_maint, N_LATENT)
    t_maint = times[maint_t]

    dt_maint = float(np.diff(t_maint).mean())
    phase    = ring_attractor_phase(Z_pca, smooth_sigma=3.0)
    vel_ac   = velocity_autocorrelation(Z_pca, dt=dt_maint, max_lag=30)
    tangling = trajectory_tangling(Z_pca, dt=dt_maint)

    var_top2 = (np.var(Z_pca[:, 0]) + np.var(Z_pca[:, 1])) / Z_pca.var(axis=0).sum()
    ring_results[subj] = {
        "phase": phase, "vel_ac": vel_ac, "tangling": tangling,
        "Z_pca": Z_pca, "t_maint": t_maint, "var_explained": var_top2,
    }
    print(f"[{subj}] Phase range: [{phase.min():.2f}, {phase.max():.2f}] rad; "
          f"Top-2 var: {var_top2:.2%}; Mean tangling: {tangling.mean():.3f}")

# Plot: 2D trajectory + phase timecourse
if ring_results:
    n_subj = len(ring_results)
    fig, axes = plt.subplots(2, n_subj, figsize=(5 * n_subj, 8))
    if n_subj == 1:
        axes = axes[:, np.newaxis]
    for col, (subj, res) in enumerate(ring_results.items()):
        t = res["t_maint"]
        sc = axes[0, col].scatter(res["Z_pca"][:, 0], res["Z_pca"][:, 1],
                                  c=t, cmap="plasma", s=20, alpha=0.8)
        axes[0, col].plot(res["Z_pca"][:, 0], res["Z_pca"][:, 1], "k-", lw=0.5, alpha=0.3)
        plt.colorbar(sc, ax=axes[0, col], label="Time (s)", fraction=0.04)
        axes[0, col].set_xlabel("PC1"); axes[0, col].set_ylabel("PC2")
        axes[0, col].set_title(f"{subj}: top-2 PCA ({res['var_explained']:.1%} var)")
        axes[0, col].set_aspect("equal")
        axes[1, col].plot(t, res["phase"], color="#4E79A7", lw=1.5)
        axes[1, col].set_xlabel("Time (s)"); axes[1, col].set_ylabel("Phase (rad)")
        axes[1, col].set_title(f"{subj}: ring attractor phase")
        axes[1, col].axhline(0, c="k", lw=0.5, ls="--")
        axes[1, col].set_ylim(-np.pi, np.pi)
    plt.suptitle("Ring attractor phase: 2-back maintenance (trial-averaged latent trajectory)",
                 fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS / "08_ring_attractor_phase.pdf", bbox_inches="tight")
    plt.show()


## Analysis 2: Cross-Temporal Generalisation (CTG)
**Hypothesis (King & Dehaene 2014):** A stable, abstract maintenance code produces off-diagonal generalisation in the train-time × test-time AUC matrix. An encoding-specific code produces only a strong diagonal.
**Prediction:** Strong diagonal during encoding (0–300 ms); extended off-diagonal generalisation during the maintenance window (300–1400 ms).


In [ ]:
# Analysis 2: cross-temporal generalisation (CTG) — classify 0-back vs 2-back
ctg_results = {}

for subj in SUBJECTS_OK:
    Z      = Z_by_subj[subj]      # (N_trials, T, k)
    labels = labels_by_subj[subj]
    times  = times_by_subj[subj]

    mask_bin   = (labels == 0) | (labels == 2)
    if mask_bin.sum() < 20:
        print(f"[{subj}] Insufficient trials for CTG, skipping")
        continue

    labels_bin = (labels[mask_bin] == 2).astype(int)
    step = 5
    Z_ctg = Z[mask_bin, ::step, :N_LATENT]   # (N_bin, T_sub, N_LATENT)
    times_sub = times[::step]

    try:
        auc_matrix = cross_temporal_generalization(
            Z_ctg, labels_bin, n_splits=3,
            rng=np.random.default_rng(42)
        )
        ctg_results[subj] = {"auc": auc_matrix, "times": times_sub}
        print(f"[{subj}] CTG {auc_matrix.shape}, "
              f"max diagonal AUC: {np.diag(auc_matrix).max():.3f}")
    except Exception as e:
        print(f"[{subj}] CTG error: {e}")

if ctg_results:
    n_subj = len(ctg_results)
    fig, axes = plt.subplots(1, n_subj, figsize=(5 * n_subj, 5))
    if n_subj == 1:
        axes = [axes]
    for ax, (subj, res) in zip(axes, ctg_results.items()):
        t_sub = res["times"]
        im = ax.imshow(res["auc"], origin="lower", aspect="auto",
                       extent=[t_sub[0], t_sub[-1], t_sub[0], t_sub[-1]],
                       cmap="RdBu_r", vmin=0.4, vmax=0.9)
        for lim in MAINT_WIN:
            ax.axhline(lim, c="k", lw=1, ls="--", alpha=0.7)
            ax.axvline(lim, c="k", lw=1, ls="--", alpha=0.7)
        ax.set_xlabel("Test time (s)"); ax.set_ylabel("Train time (s)")
        ax.set_title(f"CTG: {subj} (0-back vs 2-back)")
        plt.colorbar(im, ax=ax, label="AUC", fraction=0.04)
    plt.suptitle("Cross-temporal generalisation: WM load decoding", fontweight="bold")
    plt.tight_layout()
    plt.savefig(RESULTS / "08_ctg_matrix.pdf", bbox_inches="tight")
    plt.show()


## Analysis 3: Cross-Subject Geometry Transfer
**Hypothesis (Bernardi et al. 2020):** A universal WM geometry allows a classifier trained on subjects A–C to decode subject D after Procrustes alignment.
**Method:** LOSO; align test-subject geometry to group reference via Procrustes, then classify 0-back vs 2-back using a model trained on the remaining subjects.


In [ ]:
# Analysis 3: LOSO cross-subject geometry transfer
transfer_results = {}

if len(SUBJECTS_OK) >= 3:
    for held_out in SUBJECTS_OK:
        train_subjects = [s for s in SUBJECTS_OK if s != held_out]

        train_data, train_labels_list = [], []
        for subj in train_subjects:
            Z      = Z_by_subj[subj]       # (N, T, k)
            labels = labels_by_subj[subj]
            times  = times_by_subj[subj]
            maint_t = (times >= MAINT_WIN[0]) & (times <= MAINT_WIN[1])
            mask_bin = (labels == 0) | (labels == 2)
            if mask_bin.sum() < 5:
                continue
            Z_maint = Z[mask_bin][:, maint_t, :N_LATENT].mean(axis=1)   # (N_bin, k)
            train_data.append(Z_maint)
            train_labels_list.append((labels[mask_bin] == 2).astype(int))

        if not train_data:
            continue

        X_train = np.vstack(train_data)
        y_train = np.concatenate(train_labels_list)

        # Test set: held-out subject
        Z_ho     = Z_by_subj[held_out]
        labels_ho = labels_by_subj[held_out]
        times_ho  = times_by_subj[held_out]
        maint_ho  = (times_ho >= MAINT_WIN[0]) & (times_ho <= MAINT_WIN[1])
        mask_ho   = (labels_ho == 0) | (labels_ho == 2)
        if mask_ho.sum() < 5:
            continue
        Z_ho_maint = Z_ho[mask_ho][:, maint_ho, :N_LATENT].mean(axis=1)  # (N_ho, k)
        y_ho       = (labels_ho[mask_ho] == 2).astype(int)

        d_min = min(X_train.shape[1], Z_ho_maint.shape[1])
        proc  = procrustes_align(X_train[:len(Z_ho_maint), :d_min], Z_ho_maint[:, :d_min])
        Z_ho_aligned = proc["Y_aligned"]

        scaler  = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_train[:, :d_min])
        Z_ho_sc = scaler.transform(Z_ho_aligned)

        clf = LinearSVC(max_iter=2000, C=1.0)
        try:
            clf.fit(X_tr_sc, y_train)
            scores = clf.decision_function(Z_ho_sc)
            auc = roc_auc_score(y_ho, scores) if len(np.unique(y_ho)) > 1 else 0.5
        except Exception:
            auc = 0.5
        transfer_results[held_out] = {"auc_transfer": auc}
        print(f"  Transfer to {held_out}: AUC = {auc:.3f}")

    if transfer_results:
        fig, ax = plt.subplots(figsize=(7, 4))
        subjs = list(transfer_results.keys())
        aucs  = [transfer_results[s]["auc_transfer"] for s in subjs]
        ax.bar(subjs, aucs, color="#4E79A7", alpha=0.8)
        ax.axhline(0.5, c="r", ls="--", lw=1.5, label="Chance")
        ax.axhline(np.mean(aucs), c="k", ls="-", lw=1.5,
                   label=f"Mean AUC={np.mean(aucs):.3f}")
        ax.set_ylabel("Transfer AUC (0-back vs 2-back)")
        ax.set_title("Cross-subject geometry transfer (LOSO + Procrustes)")
        ax.legend(fontsize=8); ax.set_ylim(0, 1)
        plt.tight_layout()
        plt.savefig(RESULTS / "08_cross_subject_transfer.pdf", bbox_inches="tight")
        plt.show()
else:
    print(f"Cross-subject transfer requires ≥3 subjects, have {len(SUBJECTS_OK)}")


In [ ]:
# Consolidate results
results_dir = Path("../results")
results_08 = {
    "ring_attractor": {
        subj: {
            "var_explained_top2": float(res["var_explained"]),
            "phase_range": float(res["phase"].max() - res["phase"].min()),
            "mean_tangling": float(res["tangling"].mean()),
        }
        for subj, res in ring_results.items()
    },
    "ctg": {
        subj: {
            "max_diag_auc": float(np.diag(res["auc"]).max()),
            "mean_offdiag_auc": float(res["auc"][np.triu_indices(len(res["auc"]), k=1)].mean()),
        }
        for subj, res in ctg_results.items()
    },
    "cross_subject_transfer": {
        subj: {"auc_transfer": float(v["auc_transfer"])}
        for subj, v in transfer_results.items()
    },
}
with open(results_dir / "08_extended_results.json", "w") as f:
    json.dump(results_08, f, indent=2)
print("Results saved → results/08_extended_results.json")

if ring_results:
    print(f"  Ring attractor: mean top-2 var = {np.mean([v['var_explained'] for v in ring_results.values()]):.2%}")
if ctg_results:
    print(f"  CTG: mean max diagonal AUC = {np.mean([np.diag(v['auc']).max() for v in ctg_results.values()]):.3f}")
if transfer_results:
    print(f"  Cross-subject transfer: mean AUC = {np.mean([v['auc_transfer'] for v in transfer_results.values()]):.3f}")


## Analysis 4: Ring Attractor Quantitative Test — σ₁/σ₂ vs WM Load
**Hypothesis:** For a ring attractor, σ₁/σ₂ ≈ 1 in the top-2 PCA. If WM load strengthens ring geometry, σ₁/σ₂ should decrease toward 1.0 at 2-back. Joint test with PR: load should raise PR (more dimensions) *and* lower σ₁/σ₂ (more circular).

**References:** Compte et al. (2000) *Cereb Cortex*; Wimmer et al. (2014) *Nat Neurosci*.


In [ ]:
from sklearn.decomposition import PCA
from scipy import stats

LOAD_LEVELS_MILLER = [0, 1, 2]
WINDOWS = {
    'baseline': (-0.2, 0.0),
    'encoding': (0.0, 0.3),
    'maintenance': (0.3, 1.4),
}

# ── σ₁/σ₂ and PR joint analysis across WM loads ─────────────────────────────
ring_metric_results = {}  # {subj: {window: {load: {'sigma_ratio': ..., 'PR': ...}}}}

for subj in SUBJECTS_OK:
    Z_subj = Z_by_subj[subj]        # (N_trials, T, k)
    labels = labels_by_subj[subj]

    ring_metric_results[subj] = {}

    for win_name, (t0, t1) in WINDOWS.items():
        t_mask = (times_ref >= t0) & (times_ref <= t1)
        if t_mask.sum() < 5:
            continue

        ring_metric_results[subj][win_name] = {}

        for load in LOAD_LEVELS_MILLER:
            trial_mask = labels == load
            if trial_mask.sum() < 5:
                continue

            # Per-trial mean in window → (N_trials_load, k)
            Z = Z_subj[trial_mask][:, t_mask, :].mean(axis=1)

            n_comp = min(8, Z.shape[0] - 1, Z.shape[1])
            if n_comp < 2:
                continue

            pca = PCA(n_components=n_comp)
            pca.fit(Z)
            lambdas = pca.explained_variance_      # eigenvalues (sorted desc)
            sigmas  = np.sqrt(lambdas)             # singular values

            sigma_ratio = sigmas[0] / sigmas[1]   # σ₁/σ₂; 1 = ring, >> 1 = line/point
            PR = (lambdas.sum() ** 2) / (lambdas ** 2).sum()

            ring_metric_results[subj][win_name][load] = {
                'sigma_ratio': sigma_ratio,
                'PR': PR,
                'lambdas': lambdas,
            }

# ── Figure: σ₁/σ₂ and PR across loads and windows ───────────────────────────
n_windows = len(WINDOWS)
fig, axes = plt.subplots(2, n_windows, figsize=(5 * n_windows, 8), sharey='row')

for ax_row, metric_key, ylabel in [
    (0, 'sigma_ratio', 'σ₁/σ₂  (1 = ring)'),
    (1, 'PR', 'Participation ratio'),
]:
    for col, win_name in enumerate(WINDOWS):
        ax = axes[ax_row, col]

        # One line per subject
        colors_subj = plt.cm.Set1(np.linspace(0, 0.8, len(SUBJECTS_OK)))
        all_subj_vals = {load: [] for load in LOAD_LEVELS_MILLER}

        for subj, c in zip(SUBJECTS_OK, colors_subj):
            if win_name not in ring_metric_results[subj]:
                continue
            win_data = ring_metric_results[subj][win_name]
            loads_avail = sorted(win_data.keys())
            vals = [win_data[l][metric_key] for l in loads_avail]
            ax.plot(loads_avail, vals, 'o-', color=c, alpha=0.6, ms=6, lw=1.5, label=subj)
            for l, v in zip(loads_avail, vals):
                all_subj_vals[l].append(v)

        # Group mean ± SEM
        grp_loads = [l for l in LOAD_LEVELS_MILLER if len(all_subj_vals[l]) > 0]
        grp_means = [np.mean(all_subj_vals[l]) for l in grp_loads]
        grp_sems  = [np.std(all_subj_vals[l]) / np.sqrt(len(all_subj_vals[l]))
                     for l in grp_loads]
        ax.errorbar(grp_loads, grp_means, grp_sems,
                    fmt='ks-', ms=10, lw=2.5, capsize=6, zorder=5, label='Group')

        if metric_key == 'sigma_ratio':
            ax.axhline(1.0, c='r', ls='--', lw=1.5, alpha=0.7, label='Ring (σ₁/σ₂=1)')

        ax.set_xlabel('N-back load')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{win_name} window')
        ax.set_xticks(LOAD_LEVELS_MILLER)
        ax.set_xticklabels(['0-back', '1-back', '2-back'])
        if col == 0:
            ax.legend(fontsize=7, frameon=False)

fig.suptitle('Ring attractor geometry: σ₁/σ₂ and PR across WM loads and windows',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/08_ring_sigma_ratio.pdf', bbox_inches='tight')
plt.show()

# ── Statistical test: does σ₁/σ₂ decrease with load in maintenance window? ──
print("\n── σ₁/σ₂ in maintenance window ──")
ratios_by_load = {}
for load in LOAD_LEVELS_MILLER:
    ratios = [ring_metric_results[s]['maintenance'][load]['sigma_ratio']
              for s in SUBJECTS_OK
              if 'maintenance' in ring_metric_results[s] and load in ring_metric_results[s]['maintenance']]
    ratios_by_load[load] = ratios
    print(f"  {load}-back: σ₁/σ₂ = {np.mean(ratios):.3f} ± {np.std(ratios):.3f}  (N={len(ratios)})")

# Wilcoxon signed-rank test: 0-back vs 2-back (paired across subjects)
loads_0 = np.array([ring_metric_results[s]['maintenance'].get(0, {}).get('sigma_ratio', np.nan)
                    for s in SUBJECTS_OK])
loads_2 = np.array([ring_metric_results[s]['maintenance'].get(2, {}).get('sigma_ratio', np.nan)
                    for s in SUBJECTS_OK])
valid = ~(np.isnan(loads_0) | np.isnan(loads_2))
if valid.sum() >= 3:
    stat, p = stats.wilcoxon(loads_0[valid], loads_2[valid], alternative='greater')
    print(f"\n  Wilcoxon (0-back σ₁/σ₂ > 2-back): W={stat:.1f}, p={p:.4f}")
    d = (loads_0[valid].mean() - loads_2[valid].mean()) / np.std(loads_0[valid] - loads_2[valid])
    print(f"  Cohen's d = {d:.3f}")
    print(f"  Interpretation: {'Ring geometry strengthens with load (p<0.05)' if p < 0.05 else 'No significant load effect on ring geometry'}")

# ── Scatter: PR vs σ₁/σ₂ in maintenance (test if they're complementary) ─────
pr_vals, ratio_vals, load_colors = [], [], []
cmap = {0: '#4E79A7', 1: '#59A14F', 2: '#E15759'}
for subj in SUBJECTS_OK:
    for load in LOAD_LEVELS_MILLER:
        try:
            d = ring_metric_results[subj]['maintenance'][load]
            pr_vals.append(d['PR'])
            ratio_vals.append(d['sigma_ratio'])
            load_colors.append(cmap[load])
        except KeyError:
            pass

if pr_vals:
    fig, ax = plt.subplots(figsize=(5, 5))
    for load, color, label in [(0, '#4E79A7', '0-back'), (1, '#59A14F', '1-back'), (2, '#E15759', '2-back')]:
        idx = [i for i, c in enumerate(load_colors) if c == color]
        ax.scatter([pr_vals[i] for i in idx], [ratio_vals[i] for i in idx],
                   c=color, s=80, alpha=0.8, label=label, zorder=3)
    r, p_pearson = stats.pearsonr(pr_vals, ratio_vals)
    ax.set_xlabel('Participation ratio (PR)')
    ax.set_ylabel('σ₁/σ₂  (ring test)')
    ax.set_title(f'PR vs σ₁/σ₂: r={r:.3f}, p={p_pearson:.4f}')
    ax.legend()
    ax.axhline(1.0, c='r', ls='--', lw=1, alpha=0.5)
    plt.tight_layout()
    plt.savefig('../results/08_PR_vs_sigma_ratio.pdf', bbox_inches='tight')
    plt.show()
    print(f"\nPR vs σ₁/σ₂ correlation: r={r:.3f}, p={p_pearson:.4f}")
    print("Interpretation: " + (
        "PR and σ₁/σ₂ are inversely correlated → high-load = higher PR AND more ring-like"
        if r < -0.3 else
        "Weak correlation → PR and ring geometry are partially independent measures"
    ))

# ── Save ─────────────────────────────────────────────────────────────────────
np.savez('../results/08_ring_sigma.npz',
         subjects=np.array(SUBJECTS_OK),
         load_levels=np.array(LOAD_LEVELS_MILLER),
         sigma_ratios_maintenance=np.array([
             [ring_metric_results[s]['maintenance'].get(l, {}).get('sigma_ratio', np.nan)
              for l in LOAD_LEVELS_MILLER] for s in SUBJECTS_OK
         ]),
         pr_maintenance=np.array([
             [ring_metric_results[s]['maintenance'].get(l, {}).get('PR', np.nan)
              for l in LOAD_LEVELS_MILLER] for s in SUBJECTS_OK
         ]))
print("\nSaved → results/08_ring_sigma.npz")